In [2]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import pandas as pd

C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_data = pd.read_csv("samsum-train.csv")
test_data = pd.read_csv("samsum-test.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [4]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


## Random Sampling

In [5]:
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

## Cleaning Data

In [6]:
import re

def clean_data(text):
    text = re.sub(r"\n\r", " ", text)#lines
    text = re.sub(r"\s+", " ", text)#spaces
    text = re.sub(r"<.*?>", " ", text)#html tags
    text = text.strip().lower()
    return text

In [7]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["summary"] = val_data["summary"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [8]:
train_data.head()

,id,dialogue,summary
0,13811908,violet: hi! i came across this austin's articl...,violet sent claire austin's article.
1,13716431,pat: so does anyone know when the stream is go...,pat and lou are waiting for the stream but kev...
2,13810214,jane: jane: whaddya think? shona: this ur ti...,jane is updating her tinder profile tonight an...
3,13729823,"adam: do u have a map of paris? tom: yes, why?...",tom has a map of paris.
4,13681400,"frank: hi, how's the family? mike: great! sam'...","mike is happy, because sam's moved out. mike a..."


## Tokenize Data

In [9]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [10]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels\
    return inputs

In [11]:
train_datasets = train_data.apply(tokenize, axis=1).tolist()
val_datasets = val_data.apply(tokenize, axis=1).tolist()

## Working with our Model

In [12]:
# NLP = Generation Text

model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1152.54it/s]


In [13]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("DEVICE", device)
model.to(device)

DEVICE cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

## Training Arguments

In [14]:
training_args = TrainingArguments(
    output_dir= "./results",

    num_train_epochs= 6,
    weight_decay= 0.01,

    per_device_train_batch_size= 8,
    per_device_eval_batch_size= 8,

    eval_strategy= "epoch",
    save_strategy= "epoch",

    warmup_steps= 500
)

In [15]:
trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=train_datasets,
    eval_dataset=val_datasets,
)

## Train the model

In [ ]:
trainer.train()

## Saving the model

In [17]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.11s/it]


('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

## Use the Saved Model

In [18]:
model.from_pretrained("./saved_summary_model")
tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1173.98it/s]


T5Tokenizer(name_or_path='./saved_summary_model', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False

In [19]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 547.13it/s]


## Text the core login for Summarization

In [20]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue)

  #tokenize
  inputs = tokenizer(
      dialogue,
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  )
  #generate the summary => token ids
  model.to(device)
  targets = model.generate(
      input_ids = inputs['input_ids'],
      attention_mask = inputs['attention_mask'],
      max_length = 150,
      num_beams = 4,
      early_stopping = True
  )
  summary = tokenizer.decode(targets[0], skip_special_tokens=True)

  return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print(summary)

experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact. ensuring fairness and transparency is becoming a key area of research.
